In [2]:
import os
import torch
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

from detection import *

In [3]:
device = get_device()
device

'cuda'

In [4]:
data_path = './data/'
image_dir = os.path.join(data_path, 'images')
gt_dir = os.path.join(data_path, 'gt.csv')

In [4]:
train_loader, val_loader = prepare_dataloaders(
    image_dir,
    gt_dir,
    batch_size=32,
    split=(0.9, 0.1),
    sigma=2,
)

/home/revit3d/.cache/pypoetry/virtualenvs/vmk-qg3DzDu0-py3.13/lib/python3.13/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
hparams = {
    "sigma": 2,
    "optimizer": "AdamW",
    "lr": 5e-4,
    "weight_decay": 5e-4,
    "batch_size": 32,
    "img_size": "100x100",
    "scheduler": "OneCycleLR",
    "max_lr": 5e-3,
    "grad_clip": 1.0,
    "epochs": 50,
}

In [6]:
model = UNet(ch_mul=32)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hparams['lr'],
    weight_decay=hparams['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=hparams['max_lr'],
    steps_per_epoch=len(train_loader),
    epochs=hparams['epochs'],
)
logger = SummaryWriter(log_dir='./logs/exp_model_heatmap4')
trainer = Trainer(
    model=model.to(device),
    criterion=weighted_mse_loss,
    optimizer=optimizer,
    scheduler=scheduler,
    logger=logger,
    device=device,
)
sum(p.numel() for p in model.parameters())

1939502

In [7]:
val_losses, ema_val_losses = trainer.train(
    train_loader, val_loader, n_epochs=hparams['epochs'],
)
logger.add_hparams(hparams, {"val_loss": val_losses[-1]})
logger.close()

  0%|          | 0/50 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
torch.save(
    {
        'epoch': hparams['epochs'],
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': val_losses[-1],
    },
    './checkpoints/pretrained_model_checkpoint_0.pt',
)